In [6]:
import pandas as pd
import numpy as np

print("---  Initializing Baseline Scorecard Environment ---")

# 1. Load the prepped data structures instantly from your hard drive
X_train_base_scaled = pd.read_parquet("X_train_base_scaled.parquet", engine='fastparquet')
X_test_base_scaled  = pd.read_parquet("X_test_base_scaled.parquet", engine='fastparquet')

# 2. Extract targets and flatten back into 1D arrays
y_train = pd.read_parquet("y_train.parquet", engine='fastparquet')['target'].astype(int)
y_test  = pd.read_parquet("y_test.parquet", engine='fastparquet')['target'].astype(int)

print(f" Fast-Load Success! Workspace is fully populated.")
print(f"   ↳ Training Features Matrix Shape : {X_train_base_scaled.shape}")
print(f"   ↳ Testing Features Matrix Shape  : {X_test_base_scaled.shape}")


---  Initializing Baseline Scorecard Environment ---
 Fast-Load Success! Workspace is fully populated.
   ↳ Training Features Matrix Shape : (1062627, 88)
   ↳ Testing Features Matrix Shape  : (265657, 88)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report
import numpy as np
import pandas as pd

print("--- Stage 4: Training Logistic Regression Baseline ---")

# 1. Initialize the model with standard institutional hyper-parameters
# max_iter=1000 ensures gradient descent converges flawlessly on your 1.06M rows
# C=1.0 applies a balanced L2 regularization penalty evenly across all 88 scaled features
baseline_model = LogisticRegression(max_iter=1000, C=1.0, random_state=42, n_jobs=-1)

# 2. Fit the model strictly using the standardized training workspace
baseline_model.fit(X_train_base_scaled, y_train)

# 3. Generate risk probabilities for both train and test sets
train_probs = baseline_model.predict_proba(X_trainscaled)[:, 1]
test_probs = baseline_model.predict_proba(X_test_base_scaled)[:, 1]

# 4. Evaluate baseline performance curves
test_roc_auc = roc_auc_score(y_test, test_probs)
test_pr_auc = average_precision_score(y_test, test_probs)

print(f"\n Baseline Validation Performance Metrics:")
print(f"   ↳ Test ROC-AUC Score (Discrimination Power)   : {test_roc_auc:.4f}")
print(f"   ↳ Test PR-AUC Score  (Imbalanced Precision)   : {test_pr_auc:.4f}")

# 5. Extract class predictions using the standard 0.5 threshold
test_preds = baseline_model.predict(X_test_base_scaled)
print("\n Detailed Baseline Classification Report:")
print(classification_report(y_test, test_preds, digits=4))


# # =====================================================================
# # 🔬 CRITICAL STEP: ISOLATE THE RESIDUALS FOR YOUR DETECTIVE MODEL
# # =====================================================================
# # Residual = True Label (0 or 1) minus the continuous Probability predicted by the base model.
# # This isolates the exact direction and magnitude of the linear model's mistakes.
# X_train_residuals = y_train.values - train_probs

# print("\n✅ Baseline Model Training Complete!")
# print(f"Successfully isolated {len(X_train_residuals):,} training residuals.")
# print("The data playground is officially ready for the XGBoost Detective Track.")


--- Stage 4: Training Logistic Regression Baseline ---


d:\Model Weak-Spot Analysis\MWSA Proj Workspace\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


MemoryError: Unable to allocate 713. MiB for an array with shape (1062627, 88) and data type float64